## Import & Config

In [ ]:
!pip install pandas==1.3.5 numpy==1.26.4 --force-reinstall

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.7/4.7 MB 56.8 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.3/18.3 MB 92.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 229.9/229.9 kB 18.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 509.2/509.2 kB 29.1 MB/s eta 0:00:00
  Created wheel for pandas: filename=pandas-1.3.5-cp311-cp311-linux_x86_64.whl size=37464059 sha256=6e341ec7882be93044122bc9e73302a51fc360db26443a8d22e4d9f9702de8bc
  Stored in directory: /root/.cache/pip/wheels/8b/e7/6d/d4c288f419ab8fa07c1db6f606a2ae18ecf3dc2839d79a1c07
Successfully built pandas
  Attempting uninstall: pytz
    Found existing installation: pytz 2025.2
    Uninstalling pytz-2025.2:
      Successfully uninstalled pytz-2025.2
  Attempting uninsta

In [1]:
import pandas as pd
import numpy as np
import re

print(f'pandas version: {pd.__version__}')
print(f'numpy version: {np.__version__}')             ## numpy 버전이 1.26.4여야 glove embedding에서 gensim 불러올 때 에러가 없음..

# # DATA_ROOT = "C:/Users/AIBIZ/Desktop/sunu/Experiments/Data"
# DATA_RAW_PATH = '/content/drive/MyDrive/박민경_논문1/dataset/rawdata'
# DATA_SAVE_PATH = '/content/drive/MyDrive/박민경_논문1/dataset/prep'

DATA_PATH = "C:/Users/Bigdata64/Desktop/MK/dataset"

# raw data path
review_path = f"{DATA_PATH}/df_yelp_review.pkl"
biz_path = f"{DATA_PATH}/df_yelp_biz.pkl"
user_path = f"{DATA_PATH}/df_yelp_user.pkl"

# prep data path
review_prep_path = f"{DATA_PATH}/df_yelp_review.pkl"
biz_prep_path = f"{DATA_PATH}/df_yelp_biz.pkl"
user_prep_path = f"{DATA_PATH}/df_yelp_user.pkl"

pandas version: 1.3.5
numpy version: 1.23.5


# Business
#### Restaurants 추출 및 저장

In [3]:
df_biz = pd.read_pickle(biz_path)
print(df_biz.shape)
df_biz.head(3)

(150346, 14)


,business_id,name,address,city,state,postal_code,latitude,longitude,stars,review_count,is_open,attributes,categories,hours
0,Pns2l4eNsfO8kk83dixA6A,"Abby Rappoport, LAC, CMQ","1616 Chapala St, Ste 2",Santa Barbara,CA,93101,34.426679,-119.711197,5.0,7,0,{'ByAppointmentOnly': 'True'},"Doctors, Traditional Chinese Medicine, Naturop...",None
1,mpf3x-BjTdTEA3yCZrAYPw,The UPS Store,87 Grasso Plaza Shopping Center,Affton,MO,63123,38.551126,-90.335695,3.0,15,1,{'BusinessAcceptsCreditCards': 'True'},"Shipping Centers, Local Services, Notaries, Ma...","{'Monday': '0:0-0:0', 'Tuesday': '8:0-18:30', ..."
2,tUFrWirKiKi_TAnsVWINQQ,Target,5255 E Broadway Blvd,Tucson,AZ,85711,32.223236,-110.880452,3.5,22,0,"{'BikeParking': 'True', 'BusinessAcceptsCredit...","Department Stores, Shopping, Fashion, Home & G...","{'Monday': '8:0-22:0', 'Tuesday': '8:0-22:0', ..."


In [4]:
# 한 행에 하나의 카테고리만 존재하도록 분할.
df_biz = df_biz.assign(categories = df_biz.categories.str.split(', ')).explode('categories')
print(df_biz.shape)
df_biz.head(3)

(668695, 14)


,business_id,name,address,city,state,postal_code,latitude,longitude,stars,review_count,is_open,attributes,categories,hours
0,Pns2l4eNsfO8kk83dixA6A,"Abby Rappoport, LAC, CMQ","1616 Chapala St, Ste 2",Santa Barbara,CA,93101,34.426679,-119.711197,5.0,7,0,{'ByAppointmentOnly': 'True'},Doctors,None
0,Pns2l4eNsfO8kk83dixA6A,"Abby Rappoport, LAC, CMQ","1616 Chapala St, Ste 2",Santa Barbara,CA,93101,34.426679,-119.711197,5.0,7,0,{'ByAppointmentOnly': 'True'},Traditional Chinese Medicine,None
0,Pns2l4eNsfO8kk83dixA6A,"Abby Rappoport, LAC, CMQ","1616 Chapala St, Ste 2",Santa Barbara,CA,93101,34.426679,-119.711197,5.0,7,0,{'ByAppointmentOnly': 'True'},Naturopathic/Holistic,None


In [5]:
# 카테고리별 개수 확인.
print('Total number of categories:', len(df_biz.categories.value_counts()))
print('Top 10 categories:', '\n', df_biz.categories.value_counts()[:10])

Total number of categories: 1311
Top 10 categories: 
 Restaurants         52268
Food                27781
Shopping            24395
Home Services       14356
Beauty & Spas       14292
Nightlife           12281
Health & Medical    11890
Local Services      11198
Bars                11065
Automotive          10773
Name: categories, dtype: int64


In [6]:
#카테고리가 레스토랑인 것만 추출
df_biz_restaurant = df_biz[df_biz['categories'] == 'Restaurants']

#is_open==1인것만 추출
df_biz_restaurant = df_biz_restaurant[df_biz_restaurant['is_open']==1]

#인덱스 정렬
df_biz_restaurant = df_biz_restaurant.reset_index(drop=True)
print(df_biz_restaurant.shape)
df_biz_restaurant.head()

(34987, 14)


,business_id,name,address,city,state,postal_code,latitude,longitude,stars,review_count,is_open,attributes,categories,hours
0,MTSW4McQd7CbVtyjqoe9mw,St Honore Pastries,935 Race St,Philadelphia,PA,19107,39.955505,-75.155564,4.0,80,1,"{'RestaurantsDelivery': 'False', 'OutdoorSeati...",Restaurants,"{'Monday': '7:0-20:0', 'Tuesday': '7:0-20:0', ..."
1,CF33F8-E6oudUQ46HnavjQ,Sonic Drive-In,615 S Main St,Ashland City,TN,37015,36.269593,-87.058943,2.0,6,1,"{'BusinessParking': 'None', 'BusinessAcceptsCr...",Restaurants,"{'Monday': '0:0-0:0', 'Tuesday': '6:0-22:0', '..."
2,bBDDEgkFA1Otx9Lfe7BZUQ,Sonic Drive-In,2312 Dickerson Pike,Nashville,TN,37207,36.208102,-86.768170,1.5,10,1,"{'RestaurantsAttire': ''casual'', 'Restaurants...",Restaurants,"{'Monday': '0:0-0:0', 'Tuesday': '6:0-21:0', '..."
3,eEOYSgkmpB90uNA7lDOMRA,Vietnamese Food Truck,,Tampa Bay,FL,33602,27.955269,-82.456320,4.0,10,1,"{'Alcohol': ''none'', 'OutdoorSeating': 'None'...",Restaurants,"{'Monday': '11:0-14:0', 'Tuesday': '11:0-14:0'..."
4,il_Ro8jwPlHresjw9EGmBg,Denny's,8901 US 31 S,Indianapolis,IN,46227,39.637133,-86.127217,2.5,28,1,"{'RestaurantsReservations': 'False', 'Restaura...",Restaurants,"{'Monday': '6:0-22:0', 'Tuesday': '6:0-22:0', ..."


In [7]:
# business 데이터 우편번호, 위도, 경도 컬럼 삭제
df_biz_restaurant.drop(['is_open', 'categories'], axis=1, inplace=True)

# business 데이터 컬럼 명 변경
df_biz_restaurant = df_biz_restaurant.rename(columns={'business_id':'rest_id', 'name':'rest_name', 'stars':'rest_aver_stars', 'review_count':'rest_review_count', 'attributes':'rest_attributes'})

In [8]:
df_biz_restaurant.to_pickle(f"{biz_prep_path}")

In [9]:
df_biz_restaurant = pd.read_pickle(f"{biz_prep_path}")
df_biz_restaurant.head(5)

,rest_id,rest_name,address,city,state,postal_code,latitude,longitude,rest_aver_stars,rest_review_count,rest_attributes,hours
0,MTSW4McQd7CbVtyjqoe9mw,St Honore Pastries,935 Race St,Philadelphia,PA,19107,39.955505,-75.155564,4.0,80,"{'RestaurantsDelivery': 'False', 'OutdoorSeati...","{'Monday': '7:0-20:0', 'Tuesday': '7:0-20:0', ..."
1,CF33F8-E6oudUQ46HnavjQ,Sonic Drive-In,615 S Main St,Ashland City,TN,37015,36.269593,-87.058943,2.0,6,"{'BusinessParking': 'None', 'BusinessAcceptsCr...","{'Monday': '0:0-0:0', 'Tuesday': '6:0-22:0', '..."
2,bBDDEgkFA1Otx9Lfe7BZUQ,Sonic Drive-In,2312 Dickerson Pike,Nashville,TN,37207,36.208102,-86.768170,1.5,10,"{'RestaurantsAttire': ''casual'', 'Restaurants...","{'Monday': '0:0-0:0', 'Tuesday': '6:0-21:0', '..."
3,eEOYSgkmpB90uNA7lDOMRA,Vietnamese Food Truck,,Tampa Bay,FL,33602,27.955269,-82.456320,4.0,10,"{'Alcohol': ''none'', 'OutdoorSeating': 'None'...","{'Monday': '11:0-14:0', 'Tuesday': '11:0-14:0'..."
4,il_Ro8jwPlHresjw9EGmBg,Denny's,8901 US 31 S,Indianapolis,IN,46227,39.637133,-86.127217,2.5,28,"{'RestaurantsReservations': 'False', 'Restaura...","{'Monday': '6:0-22:0', 'Tuesday': '6:0-22:0', ..."


# Reviews
#### 컬럼명 변경

In [10]:
df_reivew = pd.read_pickle(review_path)
print(df_reivew.shape)
df_reivew.head(3)

(6990280, 9)


,review_id,user_id,business_id,stars,useful,funny,cool,text,date
0,KU_O5udG6zpxOg-VcAEodg,mh_-eMZ6K5RLWhZyISBhwA,XQfwVwDr-v0ZS3_CbbE5Xw,3.0,0,0,0,"If you decide to eat here, just be aware it is...",2018-07-07 22:09:11
1,BiTunyQ73aT9WBnpR9DZGw,OyoGAe7OKpv6SyGZT5g77Q,7ATYjTIgM3jUlt4UM3IypQ,5.0,1,0,1,I've taken a lot of spin classes over the year...,2012-01-03 15:28:18
2,saUsX_uimxRlCVr67Z4Jig,8g_iMtfSiwikVnbP2etR0A,YjUWPpI6HXG530lwP-fb2A,3.0,0,0,0,Family diner. Had the buffet. Eclectic assortm...,2014-02-05 20:30:30


In [11]:
# review 데이터 컬럼 명 변경
df_reivew = df_reivew.rename(
    columns={
        'business_id':'rest_id', 'stars':'review_stars', 'useful':'review_useful', 'funny':'review_funny',
        'cool':'review_cool', 'text':'review_text','date':'review_date'
        }
    )
print(df_reivew.shape)
df_reivew.head(3)

(6990280, 9)


,review_id,user_id,rest_id,review_stars,review_useful,review_funny,review_cool,review_text,review_date
0,KU_O5udG6zpxOg-VcAEodg,mh_-eMZ6K5RLWhZyISBhwA,XQfwVwDr-v0ZS3_CbbE5Xw,3.0,0,0,0,"If you decide to eat here, just be aware it is...",2018-07-07 22:09:11
1,BiTunyQ73aT9WBnpR9DZGw,OyoGAe7OKpv6SyGZT5g77Q,7ATYjTIgM3jUlt4UM3IypQ,5.0,1,0,1,I've taken a lot of spin classes over the year...,2012-01-03 15:28:18
2,saUsX_uimxRlCVr67Z4Jig,8g_iMtfSiwikVnbP2etR0A,YjUWPpI6HXG530lwP-fb2A,3.0,0,0,0,Family diner. Had the buffet. Eclectic assortm...,2014-02-05 20:30:30


In [12]:
df_reivew.to_pickle(f"{review_prep_path}")

In [13]:
df_reivew = pd.read_pickle(f"{review_prep_path}")
df_reivew.head(5)

,review_id,user_id,rest_id,review_stars,review_useful,review_funny,review_cool,review_text,review_date
0,KU_O5udG6zpxOg-VcAEodg,mh_-eMZ6K5RLWhZyISBhwA,XQfwVwDr-v0ZS3_CbbE5Xw,3.0,0,0,0,"If you decide to eat here, just be aware it is...",2018-07-07 22:09:11
1,BiTunyQ73aT9WBnpR9DZGw,OyoGAe7OKpv6SyGZT5g77Q,7ATYjTIgM3jUlt4UM3IypQ,5.0,1,0,1,I've taken a lot of spin classes over the year...,2012-01-03 15:28:18
2,saUsX_uimxRlCVr67Z4Jig,8g_iMtfSiwikVnbP2etR0A,YjUWPpI6HXG530lwP-fb2A,3.0,0,0,0,Family diner. Had the buffet. Eclectic assortm...,2014-02-05 20:30:30
3,AqPFMleE6RsU23_auESxiA,_7bHUi9Uuf5__HHc_Q8guQ,kxX2SOes4o-D3ZQBkiMRfA,5.0,1,0,1,"Wow! Yummy, different, delicious. Our favo...",2015-01-04 00:01:03
4,Sx8TMOWLNuJBWer-0pcmoA,bcjbaE6dDog4jkNY91ncLQ,e4Vwtrqf-wpJfwesgvdgxQ,4.0,1,0,1,Cute interior and owner (?) gave us tour of up...,2017-01-14 20:54:15


# User
#### 컬럼명 변경

In [14]:
df_user = pd.read_pickle(user_path)
print(df_user.shape)
df_user.head(3)

(1987897, 22)


,user_id,name,review_count,yelping_since,useful,funny,cool,elite,friends,fans,...,compliment_more,compliment_profile,compliment_cute,compliment_list,compliment_note,compliment_plain,compliment_cool,compliment_funny,compliment_writer,compliment_photos
0,qVc8ODYU5SZjKXVBgXdI7w,Walker,585,2007-01-25 16:47:26,7217,1259,5994,2007,"NSCy54eWehBJyZdG2iE84w, pe42u7DcCH2QmI81NX-8qA...",267,...,65,55,56,18,232,844,467,467,239,180
1,j14WgRoU_-2ZE1aw1dXrJg,Daniel,4333,2009-01-25 04:35:42,43091,13066,27281,"2009,2010,2011,2012,2013,2014,2015,2016,2017,2...","ueRPE0CX75ePGMqOFVj6IQ, 52oH4DrRvzzl8wh5UXyU0A...",3138,...,264,184,157,251,1847,7054,3131,3131,1521,1946
2,2WnXYQFK0hXEoTxPtV2zvg,Steph,665,2008-07-25 10:41:00,2086,1010,1003,"2009,2010,2011,2012,2013","LuO3Bn4f3rlhyHIaNfTlnA, j9B4XdHUhDfTKVecyWQgyA...",52,...,13,10,17,3,66,96,119,119,35,18


In [15]:
df_user.columns

Index(['user_id', 'name', 'review_count', 'yelping_since', 'useful', 'funny',
       'cool', 'elite', 'friends', 'fans', 'average_stars', 'compliment_hot',
       'compliment_more', 'compliment_profile', 'compliment_cute',
       'compliment_list', 'compliment_note', 'compliment_plain',
       'compliment_cool', 'compliment_funny', 'compliment_writer',
       'compliment_photos'],
      dtype='object')

In [16]:
# user 데이터 컬럼명 변경.
df_user = df_user.rename(
      columns={
          'name':'user_name', 'review_count':'user_review_count', 'useful':'user_useful',
          'funny':'user_funny', 'cool':'user_cool', 'elite':'user_elite','friends':'user_friends', 'fans':'user_fans', 'average_stars':'user_average_stars'
          }
      )
print(df_user.shape)
df_user.head(3)

(1987897, 22)


,user_id,user_name,user_review_count,yelping_since,user_useful,user_funny,user_cool,user_elite,user_friends,user_fans,...,compliment_more,compliment_profile,compliment_cute,compliment_list,compliment_note,compliment_plain,compliment_cool,compliment_funny,compliment_writer,compliment_photos
0,qVc8ODYU5SZjKXVBgXdI7w,Walker,585,2007-01-25 16:47:26,7217,1259,5994,2007,"NSCy54eWehBJyZdG2iE84w, pe42u7DcCH2QmI81NX-8qA...",267,...,65,55,56,18,232,844,467,467,239,180
1,j14WgRoU_-2ZE1aw1dXrJg,Daniel,4333,2009-01-25 04:35:42,43091,13066,27281,"2009,2010,2011,2012,2013,2014,2015,2016,2017,2...","ueRPE0CX75ePGMqOFVj6IQ, 52oH4DrRvzzl8wh5UXyU0A...",3138,...,264,184,157,251,1847,7054,3131,3131,1521,1946
2,2WnXYQFK0hXEoTxPtV2zvg,Steph,665,2008-07-25 10:41:00,2086,1010,1003,"2009,2010,2011,2012,2013","LuO3Bn4f3rlhyHIaNfTlnA, j9B4XdHUhDfTKVecyWQgyA...",52,...,13,10,17,3,66,96,119,119,35,18


In [17]:
df_user.to_pickle(f"{user_prep_path}")

In [18]:
df_user = pd.read_pickle(f"{user_prep_path}")
df_user.head(5)

,user_id,user_name,user_review_count,yelping_since,user_useful,user_funny,user_cool,user_elite,user_friends,user_fans,...,compliment_more,compliment_profile,compliment_cute,compliment_list,compliment_note,compliment_plain,compliment_cool,compliment_funny,compliment_writer,compliment_photos
0,qVc8ODYU5SZjKXVBgXdI7w,Walker,585,2007-01-25 16:47:26,7217,1259,5994,2007,"NSCy54eWehBJyZdG2iE84w, pe42u7DcCH2QmI81NX-8qA...",267,...,65,55,56,18,232,844,467,467,239,180
1,j14WgRoU_-2ZE1aw1dXrJg,Daniel,4333,2009-01-25 04:35:42,43091,13066,27281,"2009,2010,2011,2012,2013,2014,2015,2016,2017,2...","ueRPE0CX75ePGMqOFVj6IQ, 52oH4DrRvzzl8wh5UXyU0A...",3138,...,264,184,157,251,1847,7054,3131,3131,1521,1946
2,2WnXYQFK0hXEoTxPtV2zvg,Steph,665,2008-07-25 10:41:00,2086,1010,1003,"2009,2010,2011,2012,2013","LuO3Bn4f3rlhyHIaNfTlnA, j9B4XdHUhDfTKVecyWQgyA...",52,...,13,10,17,3,66,96,119,119,35,18
3,SZDeASXq7o05mMNLshsdIA,Gwen,224,2005-11-29 04:38:33,512,330,299,"2009,2010,2011","enx1vVPnfdNUdPho6PH_wg, 4wOcvMLtU6a9Lslggq74Vg...",28,...,4,1,6,2,12,16,26,26,10,9
4,hA5lMy-EnncsH4JoR-hFGQ,Karen,79,2007-01-05 19:40:59,29,15,7,,"PBK4q9KEEBHhFvSXCUirIw, 3FWPpM7KU1gXeOM_ZbYMbA...",1,...,1,0,0,0,1,1,0,0,0,0


# Data Merge

In [19]:
df_biz_restaurant = pd.read_pickle(f"{biz_prep_path}")
df_reivew = pd.read_pickle(f"{review_prep_path}")
df_user = pd.read_pickle(f"{user_prep_path}")

Review와 Restaurant 결합
* rest_id 기준

In [21]:
df_rest_reviews = pd.merge(df_reivew, df_biz_restaurant, on='rest_id', how = 'inner')
print(df_rest_reviews.shape)
df_rest_reviews.head(3)

(3773770, 20)


,review_id,user_id,rest_id,review_stars,review_useful,review_funny,review_cool,review_text,review_date,rest_name,address,city,state,postal_code,latitude,longitude,rest_aver_stars,rest_review_count,rest_attributes,hours
0,KU_O5udG6zpxOg-VcAEodg,mh_-eMZ6K5RLWhZyISBhwA,XQfwVwDr-v0ZS3_CbbE5Xw,3.0,0,0,0,"If you decide to eat here, just be aware it is...",2018-07-07 22:09:11,Turning Point of North Wales,1460 Bethlehem Pike,North Wales,PA,19454,40.210196,-75.223639,3.0,169,"{'NoiseLevel': 'u'average'', 'HasTV': 'False',...","{'Monday': '7:30-15:0', 'Tuesday': '7:30-15:0'..."
1,VJxlBnJmCDIy8DFG0kjSow,Iaee7y6zdSB3B-kRCo4z1w,XQfwVwDr-v0ZS3_CbbE5Xw,2.0,0,0,0,This is the second time we tried turning point...,2017-05-13 17:06:55,Turning Point of North Wales,1460 Bethlehem Pike,North Wales,PA,19454,40.210196,-75.223639,3.0,169,"{'NoiseLevel': 'u'average'', 'HasTV': 'False',...","{'Monday': '7:30-15:0', 'Tuesday': '7:30-15:0'..."
2,S6pQZQocMB1WHMjTRbt77A,ejFxLGqQcWNLdNByJlIhnQ,XQfwVwDr-v0ZS3_CbbE5Xw,4.0,2,0,1,The place is cute and the staff was very frien...,2017-08-08 00:58:18,Turning Point of North Wales,1460 Bethlehem Pike,North Wales,PA,19454,40.210196,-75.223639,3.0,169,"{'NoiseLevel': 'u'average'', 'HasTV': 'False',...","{'Monday': '7:30-15:0', 'Tuesday': '7:30-15:0'..."


user 결합
* user_id 기준

In [22]:
df_all_data = pd.merge(df_rest_reviews, df_user, on='user_id', how = 'inner')
print(df_all_data.shape)
df_all_data.head(3)

(3773763, 41)


,review_id,user_id,rest_id,review_stars,review_useful,review_funny,review_cool,review_text,review_date,rest_name,...,compliment_more,compliment_profile,compliment_cute,compliment_list,compliment_note,compliment_plain,compliment_cool,compliment_funny,compliment_writer,compliment_photos
0,KU_O5udG6zpxOg-VcAEodg,mh_-eMZ6K5RLWhZyISBhwA,XQfwVwDr-v0ZS3_CbbE5Xw,3.0,0,0,0,"If you decide to eat here, just be aware it is...",2018-07-07 22:09:11,Turning Point of North Wales,...,0,0,0,0,0,0,1,1,0,0
1,Bqn8psEmvYTO7izrnGNgqg,mh_-eMZ6K5RLWhZyISBhwA,8eDkw7CE0NKqMknPIu26fw,5.0,1,0,0,We tried this place on our first trip to New O...,2018-04-25 17:04:09,Two Chicks Cafe,...,0,0,0,0,0,0,1,1,0,0
2,78MWkzX8uQz0kDnhlhwAAg,mh_-eMZ6K5RLWhZyISBhwA,mOk3D7VczrDapNuUgLxUQw,4.0,2,0,2,So glad we stumbled upon this restaurant! It i...,2018-10-29 21:54:59,Noodle Eighty Eight,...,0,0,0,0,0,0,1,1,0,0


In [23]:
df_all_data.isna().sum()

review_id                 0
user_id                   0
rest_id                   0
review_stars              0
review_useful             0
review_funny              0
review_cool               0
review_text               0
review_date               0
rest_name                 0
address                   0
city                      0
state                     0
postal_code               0
latitude                  0
longitude                 0
rest_aver_stars           0
rest_review_count         0
rest_attributes        4071
hours                 78074
user_name                 0
user_review_count         0
yelping_since             0
user_useful               0
user_funny                0
user_cool                 0
user_elite                0
user_friends              0
user_fans                 0
user_average_stars        0
compliment_hot            0
compliment_more           0
compliment_profile        0
compliment_cute           0
compliment_list           0
compliment_note     

In [24]:
df_all_data.to_pickle(f"{DATA_PATH}/df_all_data.pkl")

In [25]:
df = pd.read_pickle(f"{DATA_PATH}/df_all_data.pkl")
print(df.shape)
df.head(3)

(3773763, 41)


,review_id,user_id,rest_id,review_stars,review_useful,review_funny,review_cool,review_text,review_date,rest_name,...,compliment_more,compliment_profile,compliment_cute,compliment_list,compliment_note,compliment_plain,compliment_cool,compliment_funny,compliment_writer,compliment_photos
0,KU_O5udG6zpxOg-VcAEodg,mh_-eMZ6K5RLWhZyISBhwA,XQfwVwDr-v0ZS3_CbbE5Xw,3.0,0,0,0,"If you decide to eat here, just be aware it is...",2018-07-07 22:09:11,Turning Point of North Wales,...,0,0,0,0,0,0,1,1,0,0
1,Bqn8psEmvYTO7izrnGNgqg,mh_-eMZ6K5RLWhZyISBhwA,8eDkw7CE0NKqMknPIu26fw,5.0,1,0,0,We tried this place on our first trip to New O...,2018-04-25 17:04:09,Two Chicks Cafe,...,0,0,0,0,0,0,1,1,0,0
2,78MWkzX8uQz0kDnhlhwAAg,mh_-eMZ6K5RLWhZyISBhwA,mOk3D7VczrDapNuUgLxUQw,4.0,2,0,2,So glad we stumbled upon this restaurant! It i...,2018-10-29 21:54:59,Noodle Eighty Eight,...,0,0,0,0,0,0,1,1,0,0


In [26]:
df.columns

Index(['review_id', 'user_id', 'rest_id', 'review_stars', 'review_useful',
       'review_funny', 'review_cool', 'review_text', 'review_date',
       'rest_name', 'address', 'city', 'state', 'postal_code', 'latitude',
       'longitude', 'rest_aver_stars', 'rest_review_count', 'rest_attributes',
       'hours', 'user_name', 'user_review_count', 'yelping_since',
       'user_useful', 'user_funny', 'user_cool', 'user_elite', 'user_friends',
       'user_fans', 'user_average_stars', 'compliment_hot', 'compliment_more',
       'compliment_profile', 'compliment_cute', 'compliment_list',
       'compliment_note', 'compliment_plain', 'compliment_cool',
       'compliment_funny', 'compliment_writer', 'compliment_photos'],
      dtype='object')

In [27]:
df_need=df[[
    'review_id', 'review_useful', 'review_text', 'review_stars', 'rest_aver_stars', 'state',
    'user_review_count', 'user_elite', 'user_friends', 'review_date', 'user_id', 'rest_id'
    ]]
print(df_need.shape)
df_need.head(3)

(3773763, 12)


,review_id,review_useful,review_text,review_stars,rest_aver_stars,state,user_review_count,user_elite,user_friends,review_date,user_id,rest_id
0,KU_O5udG6zpxOg-VcAEodg,0,"If you decide to eat here, just be aware it is...",3.0,3.0,PA,33,,"DS9QBM_NWJz1E279Zrao-A, XdXgIs4i5JFvtJf0rJlWsA...",2018-07-07 22:09:11,mh_-eMZ6K5RLWhZyISBhwA,XQfwVwDr-v0ZS3_CbbE5Xw
1,Bqn8psEmvYTO7izrnGNgqg,1,We tried this place on our first trip to New O...,5.0,4.5,LA,33,,"DS9QBM_NWJz1E279Zrao-A, XdXgIs4i5JFvtJf0rJlWsA...",2018-04-25 17:04:09,mh_-eMZ6K5RLWhZyISBhwA,8eDkw7CE0NKqMknPIu26fw
2,78MWkzX8uQz0kDnhlhwAAg,2,So glad we stumbled upon this restaurant! It i...,4.0,4.5,PA,33,,"DS9QBM_NWJz1E279Zrao-A, XdXgIs4i5JFvtJf0rJlWsA...",2018-10-29 21:54:59,mh_-eMZ6K5RLWhZyISBhwA,mOk3D7VczrDapNuUgLxUQw


In [28]:
df_need.to_pickle(f"{DATA_PATH}/df_need.pkl")

In [29]:
df_need = pd.read_pickle(f"{DATA_PATH}/df_need.pkl")
df_need.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 3773763 entries, 0 to 3773762
Data columns (total 12 columns):
 #   Column             Dtype  
---  ------             -----  
 0   review_id          object 
 1   review_useful      int64  
 2   review_text        object 
 3   review_stars       float64
 4   rest_aver_stars    float64
 5   state              object 
 6   user_review_count  int64  
 7   user_elite         object 
 8   user_friends       object 
 9   review_date        object 
 10  user_id            object 
 11  rest_id            object 
dtypes: float64(2), int64(2), object(8)
memory usage: 374.3+ MB


In [31]:
df_need['review_date'] = pd.to_datetime(df_need['review_date'])

In [30]:
df_need['state'].value_counts()

PA     836679
FL     650082
LA     461434
TN     356984
MO     273445
IN     263387
AZ     216407
NV     196156
CA     167698
NJ     139961
ID      86798
AB      54256
DE      40102
IL      30350
HI         19
XMS         5
Name: state, dtype: int64

In [32]:
df_pa=df_need[df_need['state']=='PA']
print(df_pa.shape)

# df_fl=df_need[df_need['state']=='FL']
# print(df_fl.shape)

# df_la=df_need[df_need['state']=='LA']
# print(df_la.shape)

# df_tn=df_need[df_need['state']=='TN']
# print(df_tn.shape)

# df_mo=df_need[df_need['state']=='MO']
# print(df_mo.shape)

(836679, 12)


In [33]:
# ## 주별 데이터 pkl 저장
# df_fl.to_pickle(f"{DATA_SAVE_PATH}/df_FL_data.pkl")
df_pa.to_pickle(f"{DATA_PATH}/df_PA_data.pkl")
# df_la.to_pickle(f"{DATA_SAVE_PATH}/df_LA_data.pkl")
# df_tn.to_pickle(f"{DATA_SAVE_PATH}/df_TN_data.pkl")
# df_mo.to_pickle(f"{DATA_SAVE_PATH}/df_MO_data.pkl")

In [34]:
## 상위 5개 주에 한정하여 데이터 필터링
# target_states = ['PA', 'FL', 'LA', 'TN', 'MO']
target_states = ['PA']
df_selected = df_need[df_need['state'].isin(target_states)]

In [35]:
## 주에 상관없이 전체 데이터 pkl로 저장
df_selected.to_pickle(f"{DATA_PATH}/df_selected.pkl")

In [36]:
min_date = df_selected['review_date'].min()
max_date = df_selected['review_date'].max()

print("시작 시점:", min_date.strftime("%Y-%m"))
print("종료 시점:", max_date.strftime("%Y-%m"))

시작 시점: 2005-02
종료 시점: 2022-01


In [37]:
df_selected['review_id'].nunique()

836679

# 5-core Filtering + 텍스트 전처리


In [2]:
df_selected = pd.read_pickle(f"{DATA_PATH}/df_PA_data.pkl")

In [4]:
df_selected['state'].unique()

array(['PA'], dtype=object)

In [5]:
# df_prep = df_need.copy()
df_prep = df_selected.copy()

# 텍스트 전처리
def preprocess_text(text):
    # 영문자(a-zA-Z)와 한글(가-힣)와 .만 남기고 나머지 문자 제거 후 소문자로 변환
    processed_text = re.sub(r'[^a-zA-Z가-힣\s.]', '', text).lower()
    processed_text = processed_text.replace('\n', ' ')

    return processed_text

df_prep['review_text'] = df_prep['review_text'].apply(lambda x: preprocess_text(x))
df_prep['review_text']

print(df_prep.shape)
df_prep.head(3)

(836679, 12)


,review_id,review_useful,review_text,review_stars,rest_aver_stars,state,user_review_count,user_elite,user_friends,review_date,user_id,rest_id
0,KU_O5udG6zpxOg-VcAEodg,0,if you decide to eat here just be aware it is ...,3.0,3.0,PA,33,,"DS9QBM_NWJz1E279Zrao-A, XdXgIs4i5JFvtJf0rJlWsA...",2018-07-07 22:09:11,mh_-eMZ6K5RLWhZyISBhwA,XQfwVwDr-v0ZS3_CbbE5Xw
2,78MWkzX8uQz0kDnhlhwAAg,2,so glad we stumbled upon this restaurant it is...,4.0,4.5,PA,33,,"DS9QBM_NWJz1E279Zrao-A, XdXgIs4i5JFvtJf0rJlWsA...",2018-10-29 21:54:59,mh_-eMZ6K5RLWhZyISBhwA,mOk3D7VczrDapNuUgLxUQw
3,krpCZHUj222Ha7AffGUZHQ,3,i was looking forward to a romantic dinner her...,2.0,4.0,PA,33,,"DS9QBM_NWJz1E279Zrao-A, XdXgIs4i5JFvtJf0rJlWsA...",2018-02-11 03:07:30,mh_-eMZ6K5RLWhZyISBhwA,L4kfcADLCU4T33i7Z0CkuA


In [6]:
# 전처리 후 리뷰 길이가 0인 데이터 확인 후 제거
df_prep = df_prep[df_prep['review_text'].apply(lambda x: True if len(x)>0 else False)]

print(df_prep.shape)
df_prep.head(3)

(836652, 12)


,review_id,review_useful,review_text,review_stars,rest_aver_stars,state,user_review_count,user_elite,user_friends,review_date,user_id,rest_id
0,KU_O5udG6zpxOg-VcAEodg,0,if you decide to eat here just be aware it is ...,3.0,3.0,PA,33,,"DS9QBM_NWJz1E279Zrao-A, XdXgIs4i5JFvtJf0rJlWsA...",2018-07-07 22:09:11,mh_-eMZ6K5RLWhZyISBhwA,XQfwVwDr-v0ZS3_CbbE5Xw
2,78MWkzX8uQz0kDnhlhwAAg,2,so glad we stumbled upon this restaurant it is...,4.0,4.5,PA,33,,"DS9QBM_NWJz1E279Zrao-A, XdXgIs4i5JFvtJf0rJlWsA...",2018-10-29 21:54:59,mh_-eMZ6K5RLWhZyISBhwA,mOk3D7VczrDapNuUgLxUQw
3,krpCZHUj222Ha7AffGUZHQ,3,i was looking forward to a romantic dinner her...,2.0,4.0,PA,33,,"DS9QBM_NWJz1E279Zrao-A, XdXgIs4i5JFvtJf0rJlWsA...",2018-02-11 03:07:30,mh_-eMZ6K5RLWhZyISBhwA,L4kfcADLCU4T33i7Z0CkuA


In [7]:
# 5-core 필터링
def filter_user_item(df, user_col='user_id', item_col='rest_id', min_review_nums = 5):
    while True:
      before_shape = df.shape[0]

      # 사용자당 아이템 수가 5개 이상
      user_item_counts = df.groupby(user_col)[item_col].nunique()
      df = df[df[user_col].isin(user_item_counts[user_item_counts >= min_review_nums].index)]

      # 아이템당 사용자 수가 5개 이상
      item_user_counts = df.groupby(item_col)[user_col].nunique()
      df = df[df[item_col].isin(item_user_counts[item_user_counts >= min_review_nums].index)]

      after_shape = df.shape[0]

      # 변화 없으면 종료
      if before_shape == after_shape:
          break

    return df

df_prep_5core = filter_user_item(df = df_prep)

print(df_prep_5core.shape)
df_prep_5core.head(3)

(472877, 12)


,review_id,review_useful,review_text,review_stars,rest_aver_stars,state,user_review_count,user_elite,user_friends,review_date,user_id,rest_id
0,KU_O5udG6zpxOg-VcAEodg,0,if you decide to eat here just be aware it is ...,3.0,3.0,PA,33,,"DS9QBM_NWJz1E279Zrao-A, XdXgIs4i5JFvtJf0rJlWsA...",2018-07-07 22:09:11,mh_-eMZ6K5RLWhZyISBhwA,XQfwVwDr-v0ZS3_CbbE5Xw
2,78MWkzX8uQz0kDnhlhwAAg,2,so glad we stumbled upon this restaurant it is...,4.0,4.5,PA,33,,"DS9QBM_NWJz1E279Zrao-A, XdXgIs4i5JFvtJf0rJlWsA...",2018-10-29 21:54:59,mh_-eMZ6K5RLWhZyISBhwA,mOk3D7VczrDapNuUgLxUQw
3,krpCZHUj222Ha7AffGUZHQ,3,i was looking forward to a romantic dinner her...,2.0,4.0,PA,33,,"DS9QBM_NWJz1E279Zrao-A, XdXgIs4i5JFvtJf0rJlWsA...",2018-02-11 03:07:30,mh_-eMZ6K5RLWhZyISBhwA,L4kfcADLCU4T33i7Z0CkuA


In [8]:
# 사용자별 아이템 수(max, min, mean) -> 5-core
df_num_items_per_user = df_prep_5core.groupby('user_id')['rest_id'].nunique()
_mean_user_item = df_num_items_per_user.mean()
_max_user_item = df_num_items_per_user.max()
_min_user_item = df_num_items_per_user.min()

# 아이템별 사용자 수(max, min, mean) -> 5-core
df_num_user_per_item = df_prep_5core.groupby('rest_id')['user_id'].nunique()
_mean_item_user = df_num_user_per_item.mean()
_max_item_user = df_num_user_per_item.max()
_min_item_user = df_num_user_per_item.min()

print(f'사용자별 아이템 수:, {round(_mean_user_item, 2)} (평균), {_max_user_item} (최대값), {_min_user_item} (최솟값)')
print(f'아이템별 사용자 수:, {round(_mean_item_user, 2)} (평균), {_max_item_user} (최대값), {_min_item_user} (최솟값)')

사용자별 아이템 수:, 13.65 (평균), 689 (최대값), 5 (최솟값)
아이템별 사용자 수:, 62.91 (평균), 2770 (최대값), 5 (최솟값)


In [9]:
# 사용자/아이템/총 평점 수
rating_cnt = df_prep_5core['review_stars'].count()
user_cnt = df_prep_5core['user_id'].nunique()
item_cnt = df_prep_5core['rest_id'].nunique()

# sparsity
sparsity = 1 - (rating_cnt / (user_cnt*item_cnt))*100

print(f'사용자 수: {user_cnt}')
print(f'아이템 수: {item_cnt}')
print(f'총 평점 수: {rating_cnt}')
print(f"Sparsity: {sparsity:.4f} ({sparsity * 100:.2f}%)")

사용자 수: 32927
아이템 수: 7144
총 평점 수: 472877
Sparsity: 0.7990 (79.90%)


In [10]:
# 5-core까지 전처리한 데이터 pkl로 저장
df_prep_5core.to_pickle(f'{DATA_PATH}/yelp_data.pkl', protocol=4)

In [11]:
df_prep = pd.read_pickle(f'{DATA_PATH}/yelp_data.pkl')
print(df_prep.shape)
df_prep.head(3)

(472877, 12)


,review_id,review_useful,review_text,review_stars,rest_aver_stars,state,user_review_count,user_elite,user_friends,review_date,user_id,rest_id
0,KU_O5udG6zpxOg-VcAEodg,0,if you decide to eat here just be aware it is ...,3.0,3.0,PA,33,,"DS9QBM_NWJz1E279Zrao-A, XdXgIs4i5JFvtJf0rJlWsA...",2018-07-07 22:09:11,mh_-eMZ6K5RLWhZyISBhwA,XQfwVwDr-v0ZS3_CbbE5Xw
2,78MWkzX8uQz0kDnhlhwAAg,2,so glad we stumbled upon this restaurant it is...,4.0,4.5,PA,33,,"DS9QBM_NWJz1E279Zrao-A, XdXgIs4i5JFvtJf0rJlWsA...",2018-10-29 21:54:59,mh_-eMZ6K5RLWhZyISBhwA,mOk3D7VczrDapNuUgLxUQw
3,krpCZHUj222Ha7AffGUZHQ,3,i was looking forward to a romantic dinner her...,2.0,4.0,PA,33,,"DS9QBM_NWJz1E279Zrao-A, XdXgIs4i5JFvtJf0rJlWsA...",2018-02-11 03:07:30,mh_-eMZ6K5RLWhZyISBhwA,L4kfcADLCU4T33i7Z0CkuA


# 사용자/아이템별로 리뷰 집합 생성

In [12]:
# 사용자별 모으기
from bs4 import BeautifulSoup

grouped_user_reviews = df_prep[["user_id", "review_text"]].groupby("user_id")["review_text"].sum()
grouped_user_reviews = pd.DataFrame(grouped_user_reviews)
grouped_user_reviews.rename(columns = {"review_text":"userReviews"}, inplace = True)
grouped_user_reviews['userReviews'] = grouped_user_reviews['userReviews'].apply(
    lambda txt: BeautifulSoup(txt, "html.parser").get_text()
)
grouped_user_reviews

# grouped_user_reviews['userReviews'] = grouped_user_reviews['userReviews'].apply(
#     lambda txt: re.sub(r"\.", " ", txt).replace('\n', ' ')
# )

# 인덱스를 열로 변환
grouped_user_reviews.reset_index(inplace=True)
grouped_user_reviews.rename(columns = {"user_id":"userID"}, inplace = True)
grouped_user_reviews.head(5)

,userID,userReviews
0,--4AjktZiHowEIBCMd4CZA,i love village whiskey its just a shame its al...
1,--F7iEaFFPO1UYemj-dUNw,the food and the servers are excellent. the wa...
2,--_r6E98SNIrGU7weyNxbw,i came here for a couple of lowkey drinks and ...
3,--ccVMj2PN6Z9qtdOdlung,ive never actually been into allegro because m...
4,--dVKamoZnV2vYwKtWMVVA,i dont really get why some locals love this pl...


In [13]:
yelp_user = pd.DataFrame({
    "userID": grouped_user_reviews["userID"].values,
    "userReviews": grouped_user_reviews["userReviews"].values})

print(yelp_user.shape)
yelp_user.head(3)

(32927, 2)


,userID,userReviews
0,--4AjktZiHowEIBCMd4CZA,i love village whiskey its just a shame its al...
1,--F7iEaFFPO1UYemj-dUNw,the food and the servers are excellent. the wa...
2,--_r6E98SNIrGU7weyNxbw,i came here for a couple of lowkey drinks and ...


In [ ]:
# 데이터프레임을 지정된 경로에 저장
yelp_user.to_pickle(f'{DATA_PATH}/yelp_user.pkl', protocol=4)

In [51]:
# 데이터프레임을 지정된 경로에서 불러오기
yelp_user = pd.read_pickle(f'{DATA_PATH}/yelp_user.pkl')
yelp_user.head()

,userID,userReviews
0,--4AjktZiHowEIBCMd4CZA,i love village whiskey its just a shame its al...
1,--F7iEaFFPO1UYemj-dUNw,the food and the servers are excellent. the wa...
2,--_r6E98SNIrGU7weyNxbw,i came here for a couple of lowkey drinks and ...
3,--ccVMj2PN6Z9qtdOdlung,ive never actually been into allegro because m...
4,--dVKamoZnV2vYwKtWMVVA,i dont really get why some locals love this pl...


In [ ]:
# 아이템별 모으기
grouped_item_reviews = df_prep_5core[["rest_id", "review_text"]].groupby("rest_id")["review_text"].sum()
grouped_item_reviews = pd.DataFrame(grouped_item_reviews)
grouped_item_reviews.rename(columns = {"review_text":"itemReviews"}, inplace = True)
grouped_item_reviews['itemReviews'] = grouped_item_reviews['itemReviews'].apply(
    lambda txt: BeautifulSoup(txt, "html.parser").get_text()
)

# grouped_item_reviews['itemReviews'] = grouped_item_reviews['itemReviews'].apply(
#     lambda txt: re.sub(r"\.", " ", txt).replace('\n', ' ')
# )

# 인덱스를 열로 변환
grouped_item_reviews.reset_index(inplace=True)
grouped_item_reviews.rename(columns = {"rest_id":"itemID"}, inplace = True)
grouped_item_reviews.head()

,itemID,itemReviews
0,---kPU91CF4Lq2-WlRu9Lw,great little spot and best of all byob \nlove ...
1,--epgcb7xHGuJ-4PUeSLAw,came here because i received a gift card for x...
2,-0EdehHjIQc0DtYU8QcAig,is primarily a takeout restaurant with some se...
3,-0FX23yAacC4bbLaGPvyxw,over the top excellent with service food and a...
4,-0TffRSXXIlBYVbb5AwfTg,im a big fan of indeblues collingswood locatio...


In [53]:
yelp_item = pd.DataFrame({
    "itemID": grouped_item_reviews["itemID"].values,
    "itemReviews": grouped_item_reviews["itemReviews"].values})

print(yelp_item.shape)

# 데이터프레임을 지정된 경로에 저장
yelp_item.to_pickle(f'{DATA_PATH}/yelp_item.pkl', protocol=4)

(19622, 2)


In [54]:
# 데이터프레임을 지정된 경로에서 불러오기
yelp_item = pd.read_pickle(f'{DATA_PATH}/yelp_item.pkl')
yelp_item.head()

,itemID,itemReviews
0,---kPU91CF4Lq2-WlRu9Lw,great little spot and best of all byob \nlove ...
1,--epgcb7xHGuJ-4PUeSLAw,came here because i received a gift card for x...
2,-0EdehHjIQc0DtYU8QcAig,is primarily a takeout restaurant with some se...
3,-0FX23yAacC4bbLaGPvyxw,over the top excellent with service food and a...
4,-0TffRSXXIlBYVbb5AwfTg,im a big fan of indeblues collingswood locatio...
